# Ice-cream sales trend: from a line to a quintic curve

**Finding Curve** · a worked example from the StatNex teaching guide

**The story.** An ice-cream maker has quarterly sales for ten years, 2015 to 2024. The EPA method, a way to adjust sales for the seasons, has removed the seasons from them. What is left is the trend-cycle (TC): the long-run growth and the sharp drop in 2020.

**The question.** Which curve best describes the trend, and is a more complicated curve really better?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/curve_fitting.csv` (40 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Year,Quarter,TCSI: actual,TC
2015.0,1.0,3171.0,4011.75
,2.0,4506.0,4054.15
,3.0,5467.0,4061.03
,4.0,3063.0,4107.23
2016.0,1.0,3420.0,4183.19
,2.0,4555.0,4186.51
,3.0,5640.0,4224.24
,4.0,3328.0,4365.65
2017.0,1.0,3582.0,4426.48
,2.0,4856.0,4462.27
,3.0,6039.0,4461.97
,4.0,3359.0,4496.4
2018.0,1.0,3764.0,4594.93
,2.0,5148.0,4680.55
,3.0,6151.0,4649.97
,4.0,3596.0,4725.15
2019.0,1.0,3950.0,4838.23
,2.0,5367.0,4861.18
,3.0,6385.0,4944.26
,4.0,3893.0,4843.51
2020.0,1.0,3518.0,4564.46
,2.0,5008.0,4410.28
,3.0,5832.0,4459.85
,4.0,3527.0,4633.63
2021.0,1.0,4132.0,5046.86
,2.0,5945.0,5356.24
,3.0,7117.0,5470.55
,4.0,4323.0,5536.2
2022.0,1.0,4518.0,5618.35
,2.0,6362.0,5731.17
,3.0,7609.0,5795.64
,4.0,4487.0,5846.82
2023.0,1.0,4861.0,5885.85
,2.0,6417.0,5892.86
,3.0,7850.0,5999.59
,4.0,4789.0,6121.97
2024.0,1.0,4914.0,6035.75
,2.0,6504.0,5980.25
,3.0,8055.0,6092.33
,4.0,4754.0,6165.17

These cells save `curve_fitting.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile curve_fitting.py
"""The thirteen curves of curve fitting, each an ordinary least-squares fit (statsmodels OLS).

Five are polynomials, three are a straight line in a transformed x (1/x, sqrt(x),
log x), and five are made linear by taking logs of y -- the power and exponential
curves, and the three that rise towards a ceiling K. A fit returns the coefficients
as the formula states them, and `model`, the statsmodels regression that was actually
solved: its R-squared, analysis of variance, residuals and prediction intervals all come
from there. For the last five that regression is on the transformed scale, so those
numbers describe the fit on the scale it was made on.
"""

from dataclasses import dataclass
from typing import Callable

import numpy as np
import statsmodels.api as sm


@dataclass
class Fit:
    name: str
    formula: str
    coefficients: dict  # name -> value, in the order the formula writes them
    model: object  # the statsmodels OLS results of the regression solved
    predict: Callable  # x -> y on the original scale
    back: Callable  # (lower, upper) on the fitted scale -> on the scale of y


def unchanged(lower, upper):
    return lower, upper


def ols(target, columns):
    """Least squares of target on the columns plus a constant (the constant last)."""
    design = np.column_stack([*columns, np.ones(len(target))])
    return sm.OLS(target, design).fit()


def polynomial(x, y, degree, name, formula):
    """y = a1 x + ... + ad x^d + b, solved on a centred and scaled x.

    Raw powers of x are nearly parallel columns when x is far from 0 (x^5 of a year is
    about 3e16), and least squares on them loses digits. The fit is taken on
    t = (x - c) / s instead, which spans the same polynomials and gives the same fitted
    values, and the coefficients are converted back to powers of x only for the table.
    """
    centre = x.mean()
    scale = np.abs(x - centre).max() or 1.0
    t = (x - centre) / scale
    model = ols(y, [t**p for p in range(1, degree + 1)])
    gamma = model.params
    # The polynomial in t, constant first, rewritten as a polynomial in x.
    in_t = np.polynomial.Polynomial(np.concatenate([gamma[-1:], gamma[:-1]]))
    in_x = in_t(np.polynomial.Polynomial([-centre / scale, 1.0 / scale]))
    raw = np.zeros(degree + 1)
    raw[: in_x.coef.size] = in_x.coef
    names = ["a"] if degree == 1 else [f"a{p}" for p in range(1, degree + 1)]
    coefficients = dict(zip(names + ["b"], [*raw[1:], raw[0]]))
    return Fit(name, formula, coefficients, model,
               lambda v: in_t((np.asarray(v, float) - centre) / scale), unchanged)


def straight_line_in(column, x, y, name, formula):
    """y = a f(x) + b for a fixed transformation f of x (1/x, sqrt(x), log x)."""
    model = ols(y, [column(x)])
    a, b = model.params
    return Fit(name, formula, {"a": a, "b": b}, model,
               lambda v: a * column(np.asarray(v, float)) + b, unchanged)


def fit_curve(model, x, y, bound):
    """Fit one model, named as the app names it; None when the data cannot take it."""
    if model in ("Line", "Quadratic", "Cubic", "Quartic", "Quintic"):
        degree = ["Line", "Quadratic", "Cubic", "Quartic", "Quintic"].index(model) + 1
        name = ["Line", "Quadratic", "Cubic", "Quartic", "Quintic"][degree - 1]
        formula = "y=ax+b" if degree == 1 else "y=" + "+".join(
            f"a{p}x" + (f"^{p}" if p > 1 else "") for p in range(1, degree + 1)) + "+b"
        return polynomial(x, y, degree, name, formula)
    if model == "Reciprocal":  # reciprocal: needs x != 0
        return None if (x == 0).any() else straight_line_in(
            lambda v: 1.0 / v, x, y, "Reciprocal", "y=a/x+b")
    if model == "Square root":  # square root: needs x >= 0
        return None if (x < 0).any() else straight_line_in(
            np.sqrt, x, y, "Square root", "y=a√x+b")
    if model == "Logarithm":  # logarithm: needs x > 0
        return None if (x <= 0).any() else straight_line_in(
            np.log, x, y, "Logarithm", "y=alogx+b")

    # The rest are not linear in their coefficients. Taking logs makes them so, and
    # the straight line is fitted on that scale.
    K = bound
    if model == "Power":  # power, y = b x^a: log y = log b + a log x
        if (x <= 0).any() or (y <= 0).any():
            return None
        fit = ols(np.log(y), [np.log(x)])
        a, b = fit.params[0], np.exp(fit.params[1])
        return Fit("Power", "y=bx^a", {"a": a, "b": b}, fit,
                   lambda v: b * np.asarray(v, float) ** a,
                   lambda lo, hi: (np.exp(lo), np.exp(hi)))
    if model == "Exponential":  # exponential, y = b a^x: log y = log b + x log a
        if (y <= 0).any():
            return None
        fit = ols(np.log(y), [x])
        a, b = np.exp(fit.params[0]), np.exp(fit.params[1])
        return Fit("Exponential", "y=ba^x", {"a": a, "b": b}, fit,
                   lambda v: b * a ** np.asarray(v, float),
                   lambda lo, hi: (np.exp(lo), np.exp(hi)))
    # The three curves with a ceiling K. Their transformations decrease as y rises, so
    # an interval's two ends swap over on the way back.
    if model == "Modified exponential":  # modified exponential, y = K - b a^x: log(K - y) = log b + x log a
        if (K - y <= 0).any():
            return None
        fit = ols(np.log(K - y), [x])
        a, b = np.exp(fit.params[0]), np.exp(fit.params[1])
        return Fit("Modified exponential", "y=K-ba^x", {"a": a, "b": b}, fit,
                   lambda v: K - b * a ** np.asarray(v, float),
                   lambda lo, hi: (K - np.exp(hi), K - np.exp(lo)))
    if model == "Logistic":  # logistic, y = K / (1 + b exp(-a x)): log(K/y - 1) = log b - a x
        if (y <= 0).any() or (K - y <= 0).any():
            return None
        fit = ols(np.log(K / y - 1.0), [x])
        a, b = -fit.params[0], np.exp(fit.params[1])
        return Fit("Logistic", "y=K/(1+bexp(-ax))", {"a": a, "b": b}, fit,
                   lambda v: K / (1.0 + b * np.exp(-a * np.asarray(v, float))),
                   lambda lo, hi: (K / (1 + np.exp(hi)), K / (1 + np.exp(lo))))
    if model == "Gompertz":  # Gompertz, y = K b^(a^x): log(-log(y/K)) = log(-log b) + x log a
        if (y / K <= 0).any() or (y / K >= 1).any():
            return None
        fit = ols(np.log(-np.log(y / K)), [x])
        a, b = np.exp(fit.params[0]), np.exp(-np.exp(fit.params[1]))
        return Fit("Gompertz", "y=Kb^(a^x)", {"a": a, "b": b}, fit,
                   lambda v: K * b ** (a ** np.asarray(v, float)),
                   lambda lo, hi: (K * np.exp(-np.exp(hi)), K * np.exp(-np.exp(lo))))
    raise ValueError(f"unknown model {model!r}")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
from curve_fitting import fit_curve

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
y_column = "TC"
# Blank and text cells are dropped. With no explanatory variable, x is the position in
# the series, 1, 2, 3, ..., which makes the fit a trend over time.
y = pd.to_numeric(data[y_column], errors="coerce").dropna().to_numpy(float)
x = np.arange(1, len(y) + 1, dtype=float)
n = len(y)

## Descriptive statistics

In [ ]:
# Variances and the covariance divide by n - 1. The correlation is Pearson's r between
# x and y: for a trend over time, how close the series is to a straight line.
x_mean, y_mean = x.mean(), y.mean()
x_var, y_var = x.var(ddof=1), y.var(ddof=1)
x_ss, y_ss = (n - 1) * x_var, (n - 1) * y_var
covariance = np.cov(x, y)[0, 1]
correlation = np.corrcoef(x, y)[0, 1]
basic = pd.DataFrame(
    {"X": [n, x_mean, x_ss, x_var, np.sqrt(x_var), covariance, correlation],
     y_column: [n, y_mean, y_ss, y_var, np.sqrt(y_var), np.nan, np.nan]},
    index=["n", "mean", "sum of squares", "variance", "SD", "covariance", "correlation"],
)
print(basic)

## Fitting the models

In [ ]:
# The models chosen, under the names the app gives them (Line, Quadratic to Quintic,
# Reciprocal, Square root, Logarithm, Power, Exponential, Modified exponential,
# Logistic, Gompertz). Each is an ordinary least-squares regression (statsmodels OLS),
# set up in curve_fitting.py: a curve is first made a straight line in its coefficients,
# by transforming x (reciprocal, square root, logarithm) or by taking logs of y (power,
# exponential and the three with a ceiling). Those three need the ceiling's value K,
# which the least squares do not estimate.
models = ["Line", "Quadratic", "Cubic", "Quartic", "Quintic"]
# K is set at 1.1 times the largest observation.
bound = 1.1 * y.max()
fits = []
for model in models:
    fit = fit_curve(model, x, y, bound)
    if fit is None:
        # Zero or negative values, a division by zero, or an observation above K.
        print(f"{model} cannot be fitted to these data.")
        continue
    fits.append(fit)

## Accuracy

In [ ]:
# Each fit's statsmodels results (fit.model) give these directly.
# - R-squared: the share of the variation in y the curve explains. It never falls when
#   a term is added, so a quintic always "beats" a line on it; do not pick a degree by it.
# - Adjusted R-squared: R-squared with a penalty for each coefficient, the fairer
#   comparison between models of different sizes.
# - Durbin-Watson: the residuals' squared step from one row to the next, over their
#   sum of squares. Near 2, neighbouring residuals are uncorrelated; near 0 they move
#   together, because the curve has missed a pattern or the data were smoothed
#   beforehand. Either way the P values below assume independent errors and are then
#   too small.
# For the curves fitted after taking logs (power, exponential and the three with a
# ceiling), all of these are on the log scale the fit was made on, so they cannot be
# set beside an untransformed model's.
anova_tables = []
accuracy = {}
for fit in fits:
    result = fit.model
    r2, adjusted = result.rsquared, result.rsquared_adj
    accuracy[fit.name] = [r2, adjusted, np.sqrt(r2),
                          np.sqrt(adjusted) if adjusted >= 0 else np.nan,
                          durbin_watson(result.resid)]
    # The analysis of variance: the regression's sum of squares against the residuals',
    # F = (regression mean square) / (error mean square), and its P value for "every
    # coefficient but the constant is 0". With trending data it is nearly always tiny;
    # it says the curve beats a flat line, not that it is the right curve.
    anova_tables.append(pd.DataFrame(
        [[result.ess, result.df_model, result.mse_model, result.fvalue, result.f_pvalue],
         [result.ssr, result.df_resid, result.mse_resid, np.nan, np.nan],
         [result.centered_tss, n - 1, np.nan, np.nan, np.nan]],
        index=["Regression", "Error", "Total"], columns=["SS", "df", "MS", "F", "P"]))
accuracy = pd.DataFrame(accuracy, index=["R-squared", "adjusted R-squared", "R",
                                         "adjusted R", "Durbin-Watson"])
print()
print(accuracy)

## Each model: coefficients and analysis of variance

In [ ]:
for fit, anova in zip(fits, anova_tables):
    print(f"\n{fit.name}: {fit.formula}")
    print(pd.Series(fit.coefficients).to_string())
    print(anova)

## Ljung-Box test of the residuals

In [ ]:
# Are the residuals still correlated with their own past? The same residuals the
# Durbin-Watson ratio reads (on the scale each model was fitted on, in row order),
# tested at every lag h up to min(10, n / 5): Q(h) = n(n + 2) * sum of r(k)^2 / (n - k)
# over k = 1..h, where r(k) is the residuals' autocorrelation at lag k, against
# chi-square on h degrees of freedom. A small P says a pattern is left. No parameter is
# taken off the degrees of freedom: that adjustment is for ARMA coefficients estimated
# from the same series, not for a curve's. The test reads the rows as a time order.
lags = min(10, n // 5)
if lags >= 1:
    ljung_box = []
    for fit in fits:
        test = acorr_ljungbox(fit.model.resid, lags=lags)
        ljung_box.append(pd.DataFrame({"model": fit.name, "lag": test.index, "Q": test["lb_stat"],
                                       "df": test.index, "P": test["lb_pvalue"]}))
    ljung_box = pd.concat(ljung_box, ignore_index=True)
    print()
    print(ljung_box.to_string(index=False))

## Predicted values

In [ ]:
predictions = pd.DataFrame({"No.": np.arange(1, n + 1), "X": x, y_column: y})
for fit in fits:
    predictions[fit.name] = fit.predict(x)
print()
print(predictions.to_string(index=False))

## What the result says

Five curves are fitted to the trend of 40 quarters, from a straight line to a quintic (a curve that may bend up to four times). The coefficient of determination is the share of the variation that a curve explains. It is 0.906 for the line and 0.948 for the quintic, so the extra terms add little. The line's slope is 58.1: sales grow by about \$58k every quarter.

The Durbin-Watson ratio checks whether the errors follow a pattern: a value near 2 means no pattern. It is 0.25 for the line and still only 0.47 for the quintic, and every Ljung-Box P value is far below 0.01. So all five curves leave the same pattern in their errors, such as the 2020 dip.

The bigger curves also bend oddly at the end: the quintic falls to 5968 in the last quarter, where the trend is 6165. The straight line is the most useful description.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.